In [1]:
import os
import pandas as pd
import re

csv = pd.read_csv("./output.csv")

csv.head()

,StreetName,Address,ZipCode,X_COORD,Y_COORD
0,BLACK ROCK RD,35,6903,-73.557934,41.178028
1,BLACK ROCK RD,79,6903,-73.559713,41.177148
2,BLACK ROCK RD,86,6903,-73.559469,41.176574
3,BLACK ROCK RD,99,6903,-73.560948,41.176826
4,BLACK ROCK RD,111,6903,-73.560947,41.176244


In [ ]:
suffix_df = csv.copy()
suffix_df["suffix"] = suffix_df["StreetName"].str.split().str[-1]
suffix_frequencies = suffix_df["suffix"].value_counts()
suffix_df["suffix_freq"] = suffix_df["suffix"].map(suffix_frequencies)
suffix_df = suffix_df.drop_duplicates(subset = ["suffix"])
suffix_df = suffix_df[suffix_df["suffix"].isin(["N","S","E","W","NE","NW","SE","SW"])]
suffix_df = suffix_df.sort_values(by = "suffix_freq", ascending= False)
suffix_df

,StreetName,Address,ZipCode,X_COORD,Y_COORD,suffix,suffix_freq
2969,DAVENPORT FARM LN W,105,6903,-73.530077,41.130211,W,248
283,SPRING HILL LN E,9,6903,-73.553715,41.162960,E,199
479,WHITE BIRCH RD S,12,6903,-73.599454,41.159333,S,172
233,SPRING HILL LN N,40,6903,-73.557942,41.164729,N,87


In [31]:
prefix_df = csv.copy()
prefix_df["prefix"] = prefix_df["StreetName"].str.split().str[0]
prefix_frequencies = prefix_df["prefix"].value_counts()
prefix_df["prefix_freq"] = prefix_df["prefix"].map(prefix_frequencies)
prefix_df = prefix_df.drop_duplicates(subset = ["prefix"])
prefix_df = prefix_df[prefix_df["prefix"].isin(["N","S","E","W","NE","NW","SE","SW"])]
prefix_df = prefix_df.sort_values(by = "prefix_freq", ascending= False)
prefix_df



,StreetName,Address,ZipCode,X_COORD,Y_COORD,prefix,prefix_freq
2430,W HAVILAND LN,47,6903,-73.571184,41.136382,W,358
1186,E MIDDLE PATENT RD,375,6831,-73.622433,41.149083,E,260
714,N LAKE DR,29,6903,-73.597750,41.155983,N,46
782,S LAKE DR,227,6903,-73.608449,41.154805,S,32


In [ ]:
## Check for collision where there are any prefixes or suffixes
suffix_and_prefix_df = csv.copy()
suffix_and_prefix_df["suffix"] = suffix_and_prefix_df["StreetName"].str.split().str[-1]
suffix_and_prefix_df["prefix"] = suffix_and_prefix_df["StreetName"].str.split().str[0]
suffix_and_prefix_df = suffix_and_prefix_df[
    suffix_and_prefix_df["suffix"].isin(["N","S","E","W","NE","NW","SE","SW"]) &
    suffix_and_prefix_df["prefix"].isin(["N","S","E","W","NE","NW","SE","SW"])
]

print(suffix_and_prefix_df)
#* Result shows that we don't need to worry about prefix/suffix collision

Empty DataFrame
Columns: [StreetName, Address, ZipCode, X_COORD, Y_COORD, suffix, prefix]
Index: []


In [46]:
new_df = csv.copy()

def get_directional(text):
    valid_dirs = ["N","S","E","W"]
    directional = None
    split_txt = text.split()
    directional =  split_txt[0] if split_txt[0] in valid_dirs else directional
    directional = split_txt[-1] if split_txt[-1] in valid_dirs else directional
    return directional

new_df["Directional"] = new_df["StreetName"].apply(get_directional)

def drop_directional(text):
    valid_dirs = ["N","S","E","W"]
    split_txt = text.split()
    split_txt = split_txt[1:] if split_txt[0] in valid_dirs else split_txt
    split_txt = split_txt[:-1] if split_txt[-1] in valid_dirs else split_txt
    return " ".join(split_txt)

new_df["StreetName"] = new_df["StreetName"].apply(drop_directional)

new_df[~new_df["Directional"].isna()]

,StreetName,Address,ZipCode,X_COORD,Y_COORD,Directional
233,SPRING HILL LN,40,6903,-73.557942,41.164729,N
239,SPRING HILL LN,49,6903,-73.559100,41.164191,N
262,SPRING HILL LN,50,6903,-73.557939,41.163668,N
283,SPRING HILL LN,9,6903,-73.553715,41.162960,E
293,SPRING HILL LN,21,6903,-73.553666,41.162318,E
...,...,...,...,...,...,...
28187,BROAD ST,73,6902,-73.547909,41.055878,W
28194,COMMONS PARK,100,6902,-73.540960,41.041203,N
28206,COMMONS PARK,110,6902,-73.541268,41.042056,N
28212,MIDDLE PATENT RD,44,6831,-73.625708,41.136627,E
